# This notebook is for demonstrating some example use cases of the package

### The examples below also assume that you've already generated some runs using the techniques described in the README.md 

In [ ]:
from src import BoidSimulator, BoidVisualizer, SimSaverLoader, ConfigManager, KernelReadout, NaiveReadout, COMReadout

# Generating Runs

##### I'd advise that you attempt to generate runs from the command line because thats the intended method so you're less likely to run into bugs that way. This process is outlined in `README.md`

In [ ]:
# Run this cell if you don't have a config.ini file yet! Warning, it may overwrite similarily named .ini files
# Bear in mind also that you can name your .ini file whatever you like, as long as the internals follow the same formatting

# Write a config file to the testing directory 
testing_dir = './tests/'
ConfigManager.write_default_ini(testing_dir)

In [ ]:
testing_dir = './tests'
path_to_ini = './tests/config.ini'
save_path = './tests'

# Start by initialising a ConfigManager object
cm = ConfigManager()

# Load your config file
cm.load_params_from_ini(path_to_ini)

# Initialise the simulater, passing the Config Manger object from which the simulator will retrieve its parameters
Simulator = BoidSimulator(param_loader=cm)

# Initialise SimSaverLoader passing the path in which runs will be saved as .npz files
ssl = SimSaverLoader(save_path=save_path)

In [ ]:
# Run a simulation
result = Simulator.run_simulation()

# Save the simulation when its complete
ssl.save_run(result)

# Flat Readout Example

## No chunking or memory mapping

In [ ]:
# path to some collection of .npz files
run_paths = 'PATH/TO/NPZS'

#using a method from simsaverloader which loads then npzs dictionary into memory
datas = SimSaverLoader.find_npzs(run_paths)
print(datas[0].keys())

#as you can see from the print out, all the run data has been loaded successfully

In [ ]:
# Below i initialise a NaiveReadout object

# here I'm passing two of the dictionaries retrieved in the previous cell, corresponding to replica1 and replica2
# I'm also defining the washout, the period with which the lorenz is being sucked into its regular path
nr = NaiveReadout(datas[0],datas[1],washout=1000,chunk_size=1000)

In [ ]:
#I can use that object to make predictions using the reservoir data
#but first I need to make some sort of readout

#this readout below is the most basic and just concatinates all the positions into one vector
state_vector = nr.get_reservoir_state_vectorised(nr.replica1)
state_vector2 = nr.get_reservoir_state_vectorised(nr.replica2)

#make a ridge prediction for t+0.5 (25*0.02) using the first replica split into testing and training
lookahead = 25 # in simulation steps
prediction, corr_coef = nr.ridge_prediction(state_vector,prediction_distance=lookahead)

#getting the returns so i can plot it
nr.plot_ridge_prediction(prediction,corr_coef,prediction_distance=lookahead,x_range=[2000,4000])

## Using Chunking and Memory Mapping
For very large runs or cost efficient readouts, I likely wont have enough memory to perform the calculations. Included in the readout classes is the option to use memory mapping and chunking

In [ ]:
#some of this stuff is explained in the previous section
run_paths = 'PATH/TO/NPZS'

# to do readouts and calculations using memory mapping, you have to load the data using memory mapping
datas = SimSaverLoader.find_npzs(run_paths,memory_map=True)

# everything else should be handled automatically
nr = NaiveReadout(datas[0],datas[1],washout=1000,chunk_size=1000)

In [ ]:
state_vector = nr.get_reservoir_state_vectorised(nr.replica1)

lookahead = 25 # in simulation steps
prediction, corr_coef = nr.ridge_prediction(state_vector,prediction_distance=lookahead)
nr.plot_ridge_prediction(prediction,corr_coef,prediction_distance=lookahead,x_range=[2000,4000])

# Center of Mass CoM Readout Example

In [ ]:
run_paths = 'PATH/TO/NPZS'
datas = SimSaverLoader.find_npzs(run_paths,memory_map=False)
cr = COMReadout(datas[0],datas[1],washout=1000,chunk_size=2000)

In [ ]:
state_vector_1 = cr.get_reservoir_state_vectorised(cr.replica1)
state_vector_2 = cr.get_reservoir_state_vectorised(cr.replica2)

In [ ]:
lookahead = 25 # in simulation steps
prediction,corr_coef = cr.ridge_prediction(state_vector_1,prediction_distance=lookahead)
cr.plot_ridge_prediction(prediction,corr_coef,prediction_distance=lookahead,x_range=[0,2000])

# Kernel Readout Example

In [ ]:
#some of this stuff is explained in the previous section
run_paths = 'PATH/TO/NPZS'
#using memory mapping (explained in previous section) because the kernel readouts are usually a lot more memory intensive
datas = SimSaverLoader.find_npzs(run_paths,memory_map=True)

# for the kernel readout, you need to specify the number of observation kernels you wish to generate
kr = KernelReadout(datas[0],datas[1],kernel_number=200,washout=1000,chunk_size=2000)

In [ ]:
#remember that the replica is just the dictionary of data representing a run, to perform calculations, we have to convert the data into some kind of readout vector
state_vector_1 = kr.get_reservoir_state_vectorised(kr.replica1)
state_vector_2 = kr.get_reservoir_state_vectorised(kr.replica2)

In [ ]:
# we can see how well this kernel readout predicts the lorenz
print(state_vector_2.shape)

lookahead = 25
prediction,corr_coef = kr.ridge_prediction(state_vector_2,prediction_distance=lookahead)
kr.plot_ridge_prediction(prediction,corr_coef,prediction_distance=lookahead,x_range=[0,2000])

In [ ]:
# we can also plot the consistent capacity

#note that there are a few different methods for calculating the consistent capacity
#the one below is most faithful to the methods outlined in the appendix of lymburn et al (2021)
cc, g2=kr.calc_consistency_profile(state_vector_1,state_vector_2,method='faithful')
kr.plot_consistency_profile(cc,g2,truncated_to=100)